# Catastrophe Bond Pricing Analysis

This notebook analyses tranche-level catastrophe bond pricing data collected from Artemis. The dependent variable is `spread_pct`. The main explanatory variable is `expected_loss_pct`, with additional controls for trigger type, peril type, and issuance year.

## Cell 1 — Import packages
Run this first. It loads the packages used for cleaning, EDA, visualisation, and regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## Cell 2 — Load the Artemis dataset
This cell tries common file locations. Keep the Excel file in the same folder as the notebook, or adjust `DATA_FILE`.

In [ ]:
DATA_FILE = "artemis_cat_bond_tranche_dataset_extended.xlsx"

possible_paths = [
    Path(DATA_FILE),
    Path("/mnt/data") / DATA_FILE,
]

for path in possible_paths:
    if path.exists():
        data_path = path
        break
else:
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Put it in the notebook folder or update DATA_FILE."
    )

# The main dataset is stored in the Data sheet.
df_raw = pd.read_excel(data_path, sheet_name="Data")

print(f"Loaded file: {data_path}")
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## Cell 3 — Clean column names and inspect structure
This standardises names and removes fully empty columns if Excel imported any.

In [ ]:
df = df_raw.copy()

# Clean column names.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Remove columns that are completely empty.
df = df.dropna(axis=1, how="all")

print(f"Cleaned shape: {df.shape}")
print(df.columns.tolist())
df.info()

## Cell 4 — Convert variables to correct types
The regression needs numeric `spread_pct`, `expected_loss_pct`, and `issue_year`. Categorical variables are converted to strings/categories.

In [ ]:
numeric_cols = ["issue_year", "tranche_size_m", "expected_loss_pct", "spread_pct", "spread_to_el_multiple"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

text_cols = [
    "deal_name", "issuer", "sponsor", "tranche_class", "peril_type",
    "trigger_type", "coverage_basis", "spread_basis", "validation_status",
    "regression_ready", "source_url", "notes"
]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Standardise yes/no field.
if "regression_ready" in df.columns:
    df["regression_ready"] = df["regression_ready"].str.title()

# Recalculate spread-to-expected-loss multiple to avoid relying on Excel formulas.
df["spread_to_el_multiple_calc"] = np.where(
    df["expected_loss_pct"] > 0,
    df["spread_pct"] / df["expected_loss_pct"],
    np.nan
)

df[["issue_year", "expected_loss_pct", "spread_pct", "trigger_type", "peril_type", "regression_ready"]].head()

## Cell 5 — Basic data quality checks
This checks sample size, missing values, and duplicated tranche identifiers.

In [ ]:
quality_summary = pd.DataFrame({
    "metric": [
        "Total rows",
        "Regression-ready rows",
        "Rows with spread",
        "Rows with expected loss",
        "Rows with both spread and expected loss",
        "Earliest issue year",
        "Latest issue year",
        "Unique deals",
        "Unique sponsors"
    ],
    "value": [
        len(df),
        (df["regression_ready"] == "Yes").sum() if "regression_ready" in df.columns else np.nan,
        df["spread_pct"].notna().sum(),
        df["expected_loss_pct"].notna().sum(),
        df[["spread_pct", "expected_loss_pct"]].dropna().shape[0],
        int(df["issue_year"].min()),
        int(df["issue_year"].max()),
        df["deal_name"].nunique(),
        df["sponsor"].nunique() if "sponsor" in df.columns else np.nan
    ]
})

quality_summary

## Cell 6 — Missing values table
This helps you transparently report the quality of the dataset.

In [ ]:
missing = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df))
      .sort_values("missing_pct", ascending=False)
)

missing[missing["missing_count"] > 0]

## Cell 7 — Create the regression sample
For the main empirical analysis, use only rows marked as regression-ready with non-missing spread and expected loss.

In [ ]:
reg = df.copy()

if "regression_ready" in reg.columns:
    reg = reg[reg["regression_ready"] == "Yes"].copy()

reg = reg.dropna(subset=["spread_pct", "expected_loss_pct", "issue_year", "trigger_type", "peril_type"]).copy()

# Keep economically valid rows.
reg = reg[(reg["spread_pct"] > 0) & (reg["expected_loss_pct"] > 0)].copy()

print(f"Regression sample size: {len(reg)}")
reg[["deal_name", "tranche_class", "issue_year", "expected_loss_pct", "spread_pct", "trigger_type", "peril_type"]].head()

## Cell 8 — Create broader peril categories
The raw `peril_type` field can be very detailed. For regression, broader categories avoid too many sparse dummy variables.

In [ ]:
def classify_peril(text):
    text = str(text).lower()
    if any(word in text for word in ["multi", "multiple", "named storm and earthquake", "storm and earthquake"]):
        return "Multi-peril"
    if any(word in text for word in ["hurricane", "named storm", "wind", "typhoon", "cyclone"]):
        return "Wind/Storm"
    if "earthquake" in text or "quake" in text:
        return "Earthquake"
    if "wildfire" in text or "fire" in text:
        return "Wildfire"
    if "flood" in text:
        return "Flood"
    if "mortality" in text or "pandemic" in text:
        return "Mortality/Pandemic"
    return "Other"

reg["broad_peril"] = reg["peril_type"].apply(classify_peril).astype("category")
reg["trigger_type"] = reg["trigger_type"].astype("category")
reg["issue_year_cat"] = reg["issue_year"].astype(int).astype("category")

reg[["peril_type", "broad_peril", "trigger_type"]].head(10)

# Exploratory Data Analysis

## Cell 9 — Descriptive statistics for main numeric variables

In [ ]:
main_numeric = ["spread_pct", "expected_loss_pct", "spread_to_el_multiple_calc", "tranche_size_m", "issue_year"]
reg[main_numeric].describe().T

## Cell 10 — Sample composition by issuance year
This shows whether your sample is balanced across time.

In [ ]:
year_counts = reg["issue_year"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(year_counts.index.astype(str), year_counts.values)
plt.xlabel("Issue year")
plt.ylabel("Number of tranches")
plt.title("Sample composition by issue year")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

year_counts.to_frame("n_tranches")

## Cell 11 — Sample composition by trigger type

In [ ]:
trigger_counts = reg["trigger_type"].value_counts()

plt.figure(figsize=(9, 5))
plt.bar(trigger_counts.index.astype(str), trigger_counts.values)
plt.xlabel("Trigger type")
plt.ylabel("Number of tranches")
plt.title("Sample composition by trigger type")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

trigger_counts.to_frame("n_tranches")

## Cell 12 — Sample composition by broad peril category

In [ ]:
peril_counts = reg["broad_peril"].value_counts()

plt.figure(figsize=(9, 5))
plt.bar(peril_counts.index.astype(str), peril_counts.values)
plt.xlabel("Broad peril category")
plt.ylabel("Number of tranches")
plt.title("Sample composition by peril category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

peril_counts.to_frame("n_tranches")

## Cell 13 — Distribution of expected loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(reg["expected_loss_pct"].dropna(), bins=20)
plt.xlabel("Expected loss (%)")
plt.ylabel("Number of tranches")
plt.title("Distribution of expected loss")
plt.tight_layout()
plt.show()

## Cell 14 — Distribution of spread

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(reg["spread_pct"].dropna(), bins=20)
plt.xlabel("Spread (%)")
plt.ylabel("Number of tranches")
plt.title("Distribution of spread")
plt.tight_layout()
plt.show()

## Cell 15 — Spread versus expected loss
This is the key relationship behind H1.

In [ ]:
x = reg["expected_loss_pct"]
y = reg["spread_pct"]

# Simple fitted line for visual interpretation.
slope, intercept = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = intercept + slope * x_line

plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.75)
plt.plot(x_line, y_line)
plt.xlabel("Expected loss (%)")
plt.ylabel("Spread (%)")
plt.title("Spread versus expected loss")
plt.tight_layout()
plt.show()

print(f"Visual fitted line: spread = {intercept:.3f} + {slope:.3f} * expected_loss")

## Cell 16 — Correlation matrix
This provides a simple numeric check of relationships between continuous variables.

In [ ]:
corr_vars = ["spread_pct", "expected_loss_pct", "spread_to_el_multiple_calc", "tranche_size_m", "issue_year"]
corr = reg[corr_vars].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_title("Correlation matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

corr

## Cell 17 — Average spread and expected loss by year
This helps motivate the inclusion of issuance-year controls.

In [ ]:
year_summary = (
    reg.groupby("issue_year", observed=True)
       .agg(
           n=("spread_pct", "size"),
           avg_spread_pct=("spread_pct", "mean"),
           avg_expected_loss_pct=("expected_loss_pct", "mean"),
           median_spread_pct=("spread_pct", "median")
       )
       .reset_index()
)

plt.figure(figsize=(10, 5))
plt.plot(year_summary["issue_year"], year_summary["avg_spread_pct"], marker="o", label="Average spread")
plt.plot(year_summary["issue_year"], year_summary["avg_expected_loss_pct"], marker="o", label="Average expected loss")
plt.xlabel("Issue year")
plt.ylabel("Percentage")
plt.title("Average spread and expected loss by issue year")
plt.legend()
plt.tight_layout()
plt.show()

year_summary

## Cell 18 — Spread by trigger type
This is a first descriptive check for H2.

In [ ]:
trigger_summary = (
    reg.groupby("trigger_type", observed=True)
       .agg(
           n=("spread_pct", "size"),
           avg_spread_pct=("spread_pct", "mean"),
           median_spread_pct=("spread_pct", "median"),
           avg_expected_loss_pct=("expected_loss_pct", "mean")
       )
       .sort_values("avg_spread_pct", ascending=False)
)

plt.figure(figsize=(10, 5))
reg.boxplot(column="spread_pct", by="trigger_type", rot=45)
plt.suptitle("")
plt.title("Spread by trigger type")
plt.xlabel("Trigger type")
plt.ylabel("Spread (%)")
plt.tight_layout()
plt.show()

trigger_summary

## Cell 19 — Spread by broad peril category
This is a first descriptive check for H3.

In [ ]:
peril_summary = (
    reg.groupby("broad_peril", observed=True)
       .agg(
           n=("spread_pct", "size"),
           avg_spread_pct=("spread_pct", "mean"),
           median_spread_pct=("spread_pct", "median"),
           avg_expected_loss_pct=("expected_loss_pct", "mean")
       )
       .sort_values("avg_spread_pct", ascending=False)
)

plt.figure(figsize=(10, 5))
reg.boxplot(column="spread_pct", by="broad_peril", rot=45)
plt.suptitle("")
plt.title("Spread by broad peril category")
plt.xlabel("Broad peril category")
plt.ylabel("Spread (%)")
plt.tight_layout()
plt.show()

peril_summary

## Cell 20 — Spread-to-expected-loss multiple
This ratio is useful descriptively because spread normally exceeds expected loss.

In [ ]:
multiple = reg["spread_to_el_multiple_calc"].replace([np.inf, -np.inf], np.nan).dropna()

plt.figure(figsize=(8, 5))
plt.hist(multiple, bins=25)
plt.xlabel("Spread / Expected loss")
plt.ylabel("Number of tranches")
plt.title("Distribution of spread-to-expected-loss multiple")
plt.tight_layout()
plt.show()

multiple.describe()

## Cell 21 — Outlier check
These rows are not automatically wrong, but you should inspect them because they can influence regression estimates.

In [ ]:
outliers = reg.assign(
    spread_rank=reg["spread_pct"].rank(ascending=False),
    el_rank=reg["expected_loss_pct"].rank(ascending=False),
    multiple_rank=reg["spread_to_el_multiple_calc"].rank(ascending=False)
)

cols = [
    "deal_name", "tranche_class", "issue_year", "expected_loss_pct", "spread_pct",
    "spread_to_el_multiple_calc", "trigger_type", "broad_peril", "source_url"
]

print("Highest spreads:")
display(outliers.sort_values("spread_pct", ascending=False)[cols].head(10))

print("Highest expected losses:")
display(outliers.sort_values("expected_loss_pct", ascending=False)[cols].head(10))

print("Highest spread-to-expected-loss multiples:")
display(outliers.sort_values("spread_to_el_multiple_calc", ascending=False)[cols].head(10))

# Regression analysis

## Cell 22 — Model 1: simple expected-loss model
This directly tests H1 using a simple OLS specification.

In [ ]:
model1 = smf.ols("spread_pct ~ expected_loss_pct", data=reg).fit(cov_type="HC3")
print(model1.summary())

## Cell 23 — Model 2: add trigger and peril controls
This tests whether trigger mechanism and peril category explain spread differences beyond expected loss.

In [ ]:
model2 = smf.ols(
    "spread_pct ~ expected_loss_pct + C(trigger_type) + C(broad_peril)",
    data=reg
).fit(cov_type="HC3")

print(model2.summary())

## Cell 24 — Model 3: full model with issuance-year controls
This matches the paper’s main regression logic: expected loss + trigger + peril + issuance year.

In [ ]:
model3 = smf.ols(
    "spread_pct ~ expected_loss_pct + C(trigger_type) + C(broad_peril) + C(issue_year)",
    data=reg
).fit(cov_type="HC3")

print(model3.summary())

## Cell 25 — Optional log-linear model
Prior literature often finds nonlinear pricing relationships. This model uses log spread and log expected loss.

In [ ]:
reg_log = reg[(reg["spread_pct"] > 0) & (reg["expected_loss_pct"] > 0)].copy()
reg_log["log_spread"] = np.log(reg_log["spread_pct"])
reg_log["log_expected_loss"] = np.log(reg_log["expected_loss_pct"])

model4 = smf.ols(
    "log_spread ~ log_expected_loss + C(trigger_type) + C(broad_peril) + C(issue_year)",
    data=reg_log
).fit(cov_type="HC3")

print(model4.summary())

## Cell 26 — Model comparison table
Use this table in the Results section to compare explanatory power across specifications.

In [ ]:
model_comparison = pd.DataFrame({
    "Model": ["M1: EL only", "M2: + trigger + peril", "M3: + year FE", "M4: log-linear"],
    "Dependent variable": ["spread_pct", "spread_pct", "spread_pct", "log_spread"],
    "N": [int(m.nobs) for m in [model1, model2, model3, model4]],
    "R_squared": [m.rsquared for m in [model1, model2, model3, model4]],
    "Adj_R_squared": [m.rsquared_adj for m in [model1, model2, model3, model4]],
    "AIC": [m.aic for m in [model1, model2, model3, model4]],
    "BIC": [m.bic for m in [model1, model2, model3, model4]],
})

model_comparison

## Cell 27 — Extract key coefficients for interpretation
This table is easier to interpret than the full statsmodels output.

In [ ]:
def tidy_model(model, model_name):
    result = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_error": model.bse.values,
        "p_value": model.pvalues.values,
        "ci_low": model.conf_int()[0].values,
        "ci_high": model.conf_int()[1].values,
    })
    result.insert(0, "model", model_name)
    return result

tidy_results = pd.concat([
    tidy_model(model1, "M1"),
    tidy_model(model2, "M2"),
    tidy_model(model3, "M3"),
    tidy_model(model4, "M4")
], ignore_index=True)

# Show the most important expected-loss coefficient across models.
tidy_results[tidy_results["term"].isin(["expected_loss_pct", "log_expected_loss"])]

## Cell 28 — Residual diagnostics for the main model
These plots help assess whether the linear model is reasonable.

In [ ]:
main_model = model3
fitted = main_model.fittedvalues
resid = main_model.resid

plt.figure(figsize=(8, 5))
plt.scatter(fitted, resid, alpha=0.75)
plt.axhline(0, linestyle="--")
plt.xlabel("Fitted spread (%)")
plt.ylabel("Residual")
plt.title("Residuals versus fitted values — Model 3")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(resid, bins=20)
plt.xlabel("Residual")
plt.ylabel("Number of tranches")
plt.title("Residual distribution — Model 3")
plt.tight_layout()
plt.show()

sm.qqplot(resid, line="45", fit=True)
plt.title("Q-Q plot of residuals — Model 3")
plt.tight_layout()
plt.show()

## Cell 29 — Influence check
Cook’s distance identifies observations that may strongly affect the regression.

In [ ]:
# Influence diagnostics require the non-robust fitted model object.
model3_nonrobust = smf.ols(
    "spread_pct ~ expected_loss_pct + C(trigger_type) + C(broad_peril) + C(issue_year)",
    data=reg
).fit()

influence = model3_nonrobust.get_influence()
reg_influence = reg.copy()
reg_influence["cooks_distance"] = influence.cooks_distance[0]

threshold = 4 / len(reg_influence)
print(f"Common Cook's distance threshold: {threshold:.4f}")

influential = reg_influence.sort_values("cooks_distance", ascending=False)[[
    "deal_name", "tranche_class", "issue_year", "expected_loss_pct", "spread_pct",
    "trigger_type", "broad_peril", "cooks_distance", "source_url"
]].head(10)

influential

## Cell 30 — Save cleaned analysis outputs
This creates CSV files you can use in your appendix or for reproducibility.

In [ ]:
output_dir = Path(".")

reg.to_csv(output_dir / "catbond_regression_sample_clean.csv", index=False)
year_summary.to_csv(output_dir / "catbond_year_summary.csv", index=False)
trigger_summary.to_csv(output_dir / "catbond_trigger_summary.csv")
peril_summary.to_csv(output_dir / "catbond_peril_summary.csv")
model_comparison.to_csv(output_dir / "catbond_model_comparison.csv", index=False)
tidy_results.to_csv(output_dir / "catbond_regression_coefficients.csv", index=False)

print("Saved cleaned datasets and summary tables as CSV files.")

# Suggested interpretation notes

Use the EDA and regression output to answer the hypotheses:

- **H1:** supported if `expected_loss_pct` has a positive and statistically meaningful coefficient.
- **H2:** supported if trigger-type dummy variables are jointly or individually meaningful after controlling for expected loss.
- **H3:** supported if broad peril categories differ after controlling for expected loss.
- **H4:** supported if issuance-year fixed effects add explanatory power or if year dummies show meaningful differences.

Be careful: this is an observational, manually constructed sample. Interpret coefficients as associations, not causal effects.